# Notebook for Landcover Analysis and Visualization

## Import libs

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import xarray as xr

## Load CSV containing Testset performance per cube and landcover

In [ ]:
REPO = Path.home() / "ARCEME-Vegetation-Recovery"
TABLE_DIR = REPO / "evaluation/analysis_data"
PLOT_DIR = REPO / "evaluation/paper_figures/final_plots/landcover"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(TABLE_DIR / "metrics_detail.csv")
summary = pd.read_csv(TABLE_DIR / "metrics_macro_by_fold.csv")

In [ ]:
landcover_df = data[
    (data["configuration"] == "SGEDConvLSTM RGB+NIR")
    & (data["split"] == "holdout")
    & (data["scope"] == "overall")
    & (data["landcover"] != "all")
].copy()

In [ ]:
landcover_summary = (
    landcover_df
    .groupby(["landcover", "approach"], as_index=False)
    .agg(
        NNSE_mean=("nnse", "mean"),
        NNSE_std=("nnse", "std"),
        NSE_mean=("nse", "mean"),
        NSE_std=("nse", "std"),
        MAE_mean=("mae_raw", "mean"),
        MAE_std=("mae_raw", "std"),
        MSE_mean=("mse_raw", "mean"),
        MSE_std=("mse_raw", "std"),
        Bias_mean=("bias_raw", "mean"),
        Bias_std=("bias_raw", "std"),
        n_spatial_pixels=("n_vegetation_pixels", "sum"),
        n_possible_pixel_times=("n_possible", "sum"),
        n_valid_pixel_times=("n_valid", "sum"),
        n_present=("cube_id", "nunique"),
        n_NNSE=("nnse", "count"),
        n_MAE=("mae_raw", "count"),
        n_Bias=("bias_raw", "count"),
    )
)
landcover_summary["coverage"] = landcover_summary.n_valid_pixel_times/landcover_summary.n_possible_pixel_times

In [ ]:
landcover_summary.sort_values(by="n_valid_pixel_times", ascending=False)

In [ ]:
landcover_summary[landcover_summary.approach == "Model"].sort_values(by="n_valid_pixel_times", ascending=False)

In [ ]:
landcover_summary.to_csv("landcover_summary.csv",sep=",")

In [ ]:
from pathlib import Path
import textwrap

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

PLOT_DIR = Path(PLOT_DIR)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_COLOR = "#0072B2"
PERSISTENCE_COLOR = "#D55E00"

model_lc = (
    landcover_summary[
        landcover_summary["approach"] == "Model"
    ]
    .set_index("landcover")
    .sort_values("n_possible_pixel_times", ascending=False)
)

persistence_lc = (
    landcover_summary[
        landcover_summary["approach"] == "Persistence"
    ]
    .set_index("landcover")
    .reindex(model_lc.index)
)

labels = [
    textwrap.fill(str(name).split(": ", 1)[-1], width=20)
    for name in model_lc.index
]

y = np.arange(len(model_lc))

In [ ]:
possible = model_lc["n_possible_pixel_times"].to_numpy(dtype=float)
valid = model_lc["n_valid_pixel_times"].to_numpy(dtype=float)
coverage = 100 * model_lc["coverage"].to_numpy(dtype=float)

labels_with_counts = [
    f"{label}\n(n = {int(n)} cubes)"
    for label, n in zip(labels, model_lc["n_present"])
]

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, max(4.5, 0.9 * len(labels))),
    sharey=True,
    gridspec_kw={"width_ratios": [1.6, 1]},
)

# (a) Absolute Beobachtungszahlen
ax = axes[0]

bars_possible = ax.barh(
    y - 0.17,
    possible / 1e6,
    height=0.30,
    color="#D6DEE5",
    label="Possible",
)

bars_valid = ax.barh(
    y + 0.17,
    valid / 1e6,
    height=0.30,
    color=MODEL_COLOR,
    label="Valid",
)

ax.bar_label(
    bars_possible,
    labels=[f"{int(n):,}" for n in possible],
    padding=4,
    fontsize=9,
)

ax.bar_label(
    bars_valid,
    labels=[f"{int(n):,}" for n in valid],
    padding=4,
    fontsize=9,
)

ax.set_yticks(y)
ax.set_yticklabels(labels_with_counts, fontsize=10)
ax.invert_yaxis()

ax.set_xlabel("Pixel–time observations (millions)")
ax.set_title("(a) Observation availability", loc="left", fontweight="bold")
ax.set_xlim(0, max(possible.max() / 1e6 * 1.35, 0.001))
ax.legend(frameon=False)

# (b) Coverage
ax = axes[1]

bars = ax.barh(
    y,
    coverage,
    height=0.50,
    color=MODEL_COLOR,
    alpha=0.85,
)

ax.bar_label(bars, fmt="%.1f%%", padding=5, fontsize=10)

ax.set_xlim(0, 115)
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xlabel("Valid / possible observations (%)")
ax.set_title("(b) Pooled coverage", loc="left", fontweight="bold")

for ax in axes:
    ax.set_axisbelow(True)
    ax.grid(axis="x", alpha=0.20)
    ax.tick_params(axis="y", length=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.text(
    0.5, 0.015,
    "Counts include all cube–landcover groups present, including groups "
    "excluded from metric aggregation. n = cubes containing the class.",
    ha="center",
    fontsize=9,
    color="0.4",
)

fig.tight_layout(rect=[0, 0.07, 1, 1])

fig.savefig(
    PLOT_DIR / "landcover_observation_availability.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    PLOT_DIR / "landcover_observation_availability.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
std_dev = True  

metrics = [
    ("NNSE", "n_NNSE", 0.5),
    ("NSE",  "n_NNSE", 0.0),
    ("MAE",  "n_MAE",  None),
    ("MSE",  "n_MAE",  None),
    ("Bias", "n_Bias", 0.0),
]

fig, axes = plt.subplots(
    2, 3,
    figsize=(17, max(8, 1.3 * len(labels) + 2)),
)

axes = axes.ravel()

for i, (metric, count_column, reference) in enumerate(metrics):
    ax = axes[i]

    if not np.array_equal(
        model_lc[count_column].to_numpy(),
        persistence_lc[count_column].to_numpy(),
    ):
        raise ValueError(
            f"{metric}: Model und Persistence haben unterschiedliche Cube-Zahlen."
        )

    for table, offset, color, marker, name in [
        (
            model_lc, -0.12, MODEL_COLOR, "o",
            "SGEDConvLSTM RGB+NIR",
        ),
        (
            persistence_lc, 0.12, PERSISTENCE_COLOR, "s",
            "Persistence",
        ),
    ]:
        means = table[f"{metric}_mean"].to_numpy(dtype=float)
        sds = table[f"{metric}_std"].to_numpy(dtype=float)
        positions = y + offset

        ax.plot(
            means,
            positions,
            linestyle="none",
            marker=marker,
            markersize=6,
            color=color,
            label=name,
            zorder=3,
        )

        if std_dev:
            show_sd = np.isfinite(means) & np.isfinite(sds)

            ax.errorbar(
                means[show_sd],
                positions[show_sd],
                xerr=sds[show_sd],
                fmt="none",
                ecolor=color,
                elinewidth=1.2,
                capsize=3,
                alpha=0.65,
                zorder=2,
            )

    if reference is not None:
        ax.axvline(
            reference,
            color="0.55",
            linestyle=":",
            linewidth=1.2,
            zorder=1,
        )

    counts = model_lc[count_column].to_numpy()

    ax.set_yticks(y)
    ax.set_yticklabels(
        [
            f"{label}\n(n = {int(n)})"
            for label, n in zip(labels, counts)
        ],
        fontsize=9,
    )
    ax.invert_yaxis()

    ax.set_title(
        f"({chr(97 + i)}) {metric}",
        loc="left",
        fontweight="bold",
    )
    ax.set_xlabel(f"Mean {metric}")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))

    if metric in ["MAE", "MSE"] and not std_dev:
        ax.set_xlim(left=0)

    ax.set_axisbelow(True)
    ax.grid(axis="x", alpha=0.20)
    ax.tick_params(axis="y", length=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    
axes[5].axis("off")

handles, legend_labels = axes[0].get_legend_handles_labels()
axes[5].legend(
    handles,
    legend_labels,
    loc="upper left",
    frameon=False,
    fontsize=11,
)

note = (
    "Equal cube weights within each landcover class.\n\n"
    "n = contributing cubes for the respective metric;\n"
    "identical counts for model and persistence.\n\n"
    "Higher NNSE/NSE and lower MAE/MSE are better.\n"
    "Bias closer to zero is better."
)

if std_dev:
    note += (
        "\n\nError bars: ±1 SD across cubes, not confidence intervals.\n"
        "No SD is shown where fewer than two cubes contribute."
    )

axes[5].text(
    0.02, 0.65,
    note,
    transform=axes[5].transAxes,
    va="top",
    fontsize=10,
    color="0.35",
    linespacing=1.5,
)

fig.tight_layout()

suffix = "with_sd" if std_dev else "means_only"

fig.savefig(
    PLOT_DIR / f"landcover_metric_comparison_{suffix}.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    PLOT_DIR / f"landcover_metric_comparison_{suffix}.pdf",
    bbox_inches="tight",
)

plt.show()